# Modelo GLM: Índice de Progresismo

Este notebook ajusta modelos GLM recursivos para predecir el Índice de Progresismo en Generales y Ballotage.

**Proceso:**
1. Cargar datos y definir variables
2. Ajustar modelo inicial con todas las variables
3. Filtrar variables significativas (p < 0.05)
4. Re-ajustar modelo solo con significativas
5. Repetir hasta convergencia (mismas variables significativas)
6. Reportar: AIC, BIC, F, p-valor, Deviance explicada, variables significativas

In [8]:
# ============================================================
# IMPORTACIONES
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [9]:
# ============================================================
# CARGAR DATOS
# ============================================================

Ruta_Datos = (
    'C:/Users/Patricio/Documents/Codigo/Python/'
    'Investigacion/Tesis/Data/Bases definitivas/'
)

df_Generales = pd.read_excel(Ruta_Datos + 'Generales.xlsx')
df_Ballotage = pd.read_excel(Ruta_Datos + 'Ballotage.xlsx')

print(f"Generales: {len(df_Generales)} observaciones")
print(f"Ballotage: {len(df_Ballotage)} observaciones")

Generales: 2786 observaciones
Ballotage: 1254 observaciones


In [10]:
# ============================================================
# DEFINIR VARIABLES
# ============================================================

# Variable dependiente.
Variable_Dependiente = 'Indice_Progresismo'

# Variables independientes (todas excepto la dependiente).
Variables_Independientes = [
    'Conservative_Cluster',
    'Progressive_Cluster',
    'Edad',
    'Genero_Masculino',
    'Genero_Otro',
    'Region_CABA',
    'Region_Centro',
    'Region_Cuyo',
    'Region_Norte',
    'Region_Patagonia',
    'Voto_2019_JL_Espert',
    'Voto_2019_J_Gomez_Centurion',
    'Voto_2019_Mauricio_Macri',
    'Voto_2019_Nicolas_Del_Caño',
    'Voto_2019_Roberto_Lavagna',
    'Categoria_PASO_2023_Left_Wing',
    'Categoria_PASO_2023_Moderate_Right_A',
    'Categoria_PASO_2023_Moderate_Right_B',
    'Categoria_PASO_2023_Right_Wing_Libertarian',
    'Categoria_PASO_2023_Centre',
    'Autopercepcion_Izq_Der',
    'Autopercepcion_Con_Pro',
    'Autopercepcion_Per_Antiper',
    'Cercania_Massa',
    'Cercania_Milei',
    'Influencia_Redes',
    'Red_Social_Facebook',
    'Red_Social_Instagram',
    'Red_Social_Threads',
    'Red_Social_Tiktok',
    'Red_Social_Youtube',
    'Red_Social_Whatsapp',
    'Red_Social_Telegram',
    'Influencia_Prensa',
    'Medios_Prensa_Prensa_Obrera',
    'Medios_Prensa_Diario_Universal',
    'Medios_Prensa_Popular',
    'Medios_Prensa_Izquierda_Diario',
    'Medios_Prensa_Clarin',
    'Medios_Prensa_Perfil',
    'Medios_Prensa_Pagina_12',
    'Medios_Prensa_Infobae',
    'Medios_Prensa_El_Cronista',
    'Medios_Prensa_La_Nacion',
    'Medios_Prensa_Tiempo_Argentino',
    'Indice_Positividad',
    'Indice_Progresismo_Tiempo',
    'Indice_Conservadurismo',
    'Indice_Conservadurismo_Tiempo'
]

In [11]:
# ============================================================
# FUNCIÓN: AJUSTAR GLM RECURSIVO
# ============================================================

def Ajustar_GLM_Recursivo(
    df,
    Variable_Dependiente,
    Variables_Independientes,
    Familia,
    Nombre_Eleccion
):

    """
    Ajusta modelo GLM recursivamente hasta convergencia.
    
    Parámetros:
    - df: DataFrame con los datos.
    - Variable_Dependiente: nombre de la variable dependiente.
    - Variables_Independientes: lista de variables 
      independientes.
    - Familia: familia de distribución del GLM.
    - Nombre_Eleccion: string identificador.
    
    Retorna:
    - Diccionario con modelo final y estadísticos.
    
    """

    print(f"\n{'='*60}")
    print(f"GLM RECURSIVO: {Nombre_Eleccion}")
    print(f"{'='*60}\n")
    
    # Preparar datos eliminando NaN.
    Columnas_Necesarias = (
        [Variable_Dependiente] + Variables_Independientes
    )
    
    df_Limpio = df[Columnas_Necesarias].dropna()
    
    print(
        f"Observaciones válidas: {len(df_Limpio)} "
        f"de {len(df)}"
    )
    
    # Inicializar variables para iteración.
    Variables_Actuales = Variables_Independientes.copy()
    Iteracion = 0
    Variables_Anteriores = []
    
    while True:
        Iteracion += 1
        print(f"\n{'─'*60}")
        print(
            f"Iteración {Iteracion}: "
            f"{len(Variables_Actuales)} variables"
        )
        print(f"{'─'*60}")
        
        # Preparar X e y.
        X = df_Limpio[Variables_Actuales].copy()
        y = df_Limpio[Variable_Dependiente]
        
        # Convertir booleanos a int para evitar errores.
        for col in X.columns:
            if X[col].dtype == bool:
                X[col] = X[col].astype(int)
        
        # Agregar constante.
        X_Con_Constante = sm.add_constant(X)
        
        # Ajustar modelo.
        Modelo = sm.GLM(
            y,
            X_Con_Constante,
            family=Familia
        ).fit()
        
        # Extraer p-valores.
        P_Valores = Modelo.pvalues
        
        # Filtrar variables significativas.
        Variables_Significativas = [
            var for var in Variables_Actuales
            if P_Valores[var] < 0.05
        ]
        
        print(
            f"Variables significativas (p<0.05): "
            f"{len(Variables_Significativas)}"
        )
        
        # Verificar convergencia.
        if (
            set(Variables_Significativas) == 
            set(Variables_Anteriores)
        ):
            print("\n✓ Convergencia alcanzada")
            break
        
        if len(Variables_Significativas) == 0:
            print(
                "\n⚠ No hay variables significativas. "
                "Deteniendo."
            )
            break
        
        # Actualizar para siguiente iteración.
        Variables_Anteriores = Variables_Actuales.copy()
        Variables_Actuales = Variables_Significativas.copy()
    
    # Modelo final.
    print(f"\n{'='*60}")
    print("MODELO FINAL")
    print(f"{'='*60}\n")
    
    # Calcular estadísticos finales.
    AIC = Modelo.aic
    BIC = Modelo.bic
    
    # Calcular Deviance explicada.
    Deviance_Nulo = Modelo.null_deviance
    Deviance_Residual = Modelo.deviance
    Deviance_Explicada = (
        1 - (Deviance_Residual / Deviance_Nulo)
    )
    
    # Calcular F-estadístico aproximado.
    N = len(df_Limpio)
    K = len(Variables_Actuales)
    
    F_Estadistico = (
        ((Deviance_Nulo - Deviance_Residual) / K) /
        (Deviance_Residual / (N - K - 1))
    )
    
    P_Valor_F = (
        1 - stats.f.cdf(F_Estadistico, K, N - K - 1)
    )
    
    # Imprimir resultados.
    print(f"N observaciones: {N}")
    print(f"Variables en modelo final: {K}")
    print(f"\nAIC: {AIC:.2f}")
    print(f"BIC: {BIC:.2f}")
    print(f"F-estadístico: {F_Estadistico:.4f}")
    print(f"P-valor (F): {P_Valor_F:.6f}")
    print(f"Deviance explicada: {Deviance_Explicada:.4f}")
    print(
        f"(equivalente a R² = "
        f"{Deviance_Explicada*100:.2f}%)"
    )
    
    print(f"\n{'─'*60}")
    print("VARIABLES SIGNIFICATIVAS FINALES:")
    print(f"{'─'*60}\n")
    
    # Ordenar por p-valor.
    Resultados_Vars = []
    
    for var in Variables_Actuales:
        Coef = Modelo.params[var]
        P_Val = Modelo.pvalues[var]
        IC = Modelo.conf_int().loc[var]
        
        Resultados_Vars.append({
            'Variable': var,
            'Coeficiente': Coef,
            'P_Valor': P_Val,
            'IC_Inferior': IC[0],
            'IC_Superior': IC[1]
        })
    
    df_Resultados = pd.DataFrame(Resultados_Vars)
    df_Resultados = df_Resultados.sort_values('P_Valor')
    
    for idx, fila in df_Resultados.iterrows():
        print(
            f"{fila['Variable']:<45} "
            f"β={fila['Coeficiente']:8.4f}  "
            f"p={fila['P_Valor']:.6f}  "
            f"IC=[{fila['IC_Inferior']:7.3f}, "
            f"{fila['IC_Superior']:7.3f}]"
        )
    
    # Retornar diccionario con resultados.
    return {
        'Modelo': Modelo,
        'Eleccion': Nombre_Eleccion,
        'N_Observaciones': N,
        'N_Variables': K,
        'AIC': AIC,
        'BIC': BIC,
        'F_Estadistico': F_Estadistico,
        'P_Valor_F': P_Valor_F,
        'Deviance_Explicada': Deviance_Explicada,
        'Variables_Significativas': Variables_Actuales,
        'Resultados_Variables': df_Resultados,
        'N_Iteraciones': Iteracion
    }

In [12]:
# ============================================================
# AJUSTAR MODELO: GENERALES
# ============================================================

# Familia Gaussiana (distribución normal con link identity).
Familia_GLM = sm.families.Gaussian(
    sm.families.links.Identity()
)

Resultado_Generales = Ajustar_GLM_Recursivo(
    df=df_Generales,
    Variable_Dependiente=Variable_Dependiente,
    Variables_Independientes=Variables_Independientes,
    Familia=Familia_GLM,
    Nombre_Eleccion='GENERALES'
)


GLM RECURSIVO: GENERALES

Observaciones válidas: 2763 de 2786

────────────────────────────────────────────────────────────
Iteración 1: 49 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 24

────────────────────────────────────────────────────────────
Iteración 2: 24 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 23

────────────────────────────────────────────────────────────
Iteración 3: 23 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 23

────────────────────────────────────────────────────────────
Iteración 4: 23 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 23

✓ Convergencia alcanzada

MODELO FINAL

N observaciones: 2763
Variables en modelo final: 23

AIC: 3643.23
BIC: -21109.72
F-estadístico: 185.6784
P-valor (F): 0.000000
Deviance explicada: 0

In [13]:
# ============================================================
# AJUSTAR MODELO: BALLOTAGE
# ============================================================

Resultado_Ballotage = Ajustar_GLM_Recursivo(
    df=df_Ballotage,
    Variable_Dependiente=Variable_Dependiente,
    Variables_Independientes=Variables_Independientes,
    Familia=Familia_GLM,
    Nombre_Eleccion='BALLOTAGE'
)


GLM RECURSIVO: BALLOTAGE

Observaciones válidas: 1249 de 1254

────────────────────────────────────────────────────────────
Iteración 1: 49 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 19

────────────────────────────────────────────────────────────
Iteración 2: 19 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 19

────────────────────────────────────────────────────────────
Iteración 3: 19 variables
────────────────────────────────────────────────────────────
Variables significativas (p<0.05): 19

✓ Convergencia alcanzada

MODELO FINAL

N observaciones: 1249
Variables en modelo final: 19

AIC: 1732.25
BIC: -8479.42
F-estadístico: 114.1635
P-valor (F): 0.000000
Deviance explicada: 0.6383
(equivalente a R² = 63.83%)

────────────────────────────────────────────────────────────
VARIABLES SIGNIFICATIVAS FINALES:
──────────────────────────────────────────────────────────

In [14]:
# ============================================================
# RESUMEN COMPARATIVO
# ============================================================

print(f"\n\n{'='*70}")
print(
    f"RESUMEN COMPARATIVO: "
    f"{Variable_Dependiente}"
)
print(f"{'='*70}\n")

Resumen_Comparativo = pd.DataFrame([
    {
        'Elección': 'Generales',
        'N': Resultado_Generales['N_Observaciones'],
        'Variables': Resultado_Generales['N_Variables'],
        'AIC': Resultado_Generales['AIC'],
        'BIC': Resultado_Generales['BIC'],
        'F': Resultado_Generales['F_Estadistico'],
        'p-valor': Resultado_Generales['P_Valor_F'],
        'Dev. Expl.': (
            Resultado_Generales['Deviance_Explicada']
        ),
        'Iteraciones': Resultado_Generales['N_Iteraciones']
    },
    {
        'Elección': 'Ballotage',
        'N': Resultado_Ballotage['N_Observaciones'],
        'Variables': Resultado_Ballotage['N_Variables'],
        'AIC': Resultado_Ballotage['AIC'],
        'BIC': Resultado_Ballotage['BIC'],
        'F': Resultado_Ballotage['F_Estadistico'],
        'p-valor': Resultado_Ballotage['P_Valor_F'],
        'Dev. Expl.': (
            Resultado_Ballotage['Deviance_Explicada']
        ),
        'Iteraciones': Resultado_Ballotage['N_Iteraciones']
    }
])

print(Resumen_Comparativo.to_string(index=False))

print(f"\n{'─'*70}")
print("Variables significativas en GENERALES:")
print(f"{'─'*70}")
for var in Resultado_Generales['Variables_Significativas']:
    print(f"  • {var}")

print(f"\n{'─'*70}")
print("Variables significativas en BALLOTAGE:")
print(f"{'─'*70}")
for var in Resultado_Ballotage['Variables_Significativas']:
    print(f"  • {var}")



RESUMEN COMPARATIVO: Indice_Progresismo

 Elección    N  Variables         AIC           BIC          F      p-valor  Dev. Expl.  Iteraciones
Generales 2763         23 3643.229639 -21109.724531 185.678446 1.110223e-16    0.609250            4
Ballotage 1249         19 1732.247506  -8479.421903 114.163495 1.110223e-16    0.638328            3

──────────────────────────────────────────────────────────────────────
Variables significativas en GENERALES:
──────────────────────────────────────────────────────────────────────
  • Conservative_Cluster
  • Edad
  • Genero_Masculino
  • Voto_2019_JL_Espert
  • Voto_2019_J_Gomez_Centurion
  • Voto_2019_Mauricio_Macri
  • Categoria_PASO_2023_Moderate_Right_A
  • Categoria_PASO_2023_Moderate_Right_B
  • Categoria_PASO_2023_Right_Wing_Libertarian
  • Categoria_PASO_2023_Centre
  • Autopercepcion_Izq_Der
  • Autopercepcion_Con_Pro
  • Autopercepcion_Per_Antiper
  • Cercania_Massa
  • Cercania_Milei
  • Influencia_Redes
  • Medios_Prensa_Izquierda_